# 05 - SHAP Analysis
## Model Explainability & Feature Importance

**Tujuan:** Memahami kontribusi setiap fitur terhadap prediksi model menggunakan SHAP values.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
import joblib

print('Libraries loaded')

## 1. Load Model & Data

In [ ]:
model = joblib.load('../api/models/artifacts/rf_model_v1.pkl')
pipeline = joblib.load('../api/models/artifacts/preprocessing_pipeline.pkl')

df = pd.read_csv('../data/sample/sample_input.csv')
print(f'Model: {type(model).__name__}')
print(f'Data shape: {df.shape}')

## 2. SHAP Explainer

In [ ]:
feature_cols = ['storage_temp', 'ph', 'storage_time', 'pasteurization_temp', 'tpc', 'grading_delta_hours', 'shift_Pagi', 'shift_Siang']

df_sample = df.copy()
df_sample['tpc'] = df_sample['tpc'].fillna(df_sample['tpc'].median())
df_sample['grading_delta_hours'] = df_sample['grading_delta_hours'].fillna(df_sample['grading_delta_hours'].median())
df_sample['shift_Pagi'] = (df_sample['shift'] == 'Pagi').astype(int)
df_sample['shift_Siang'] = (df_sample['shift'] == 'Siang').astype(int)

X = df_sample[feature_cols]
X_scaled = pipeline.transform(X)

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_scaled)

print(f'SHAP values shape: {np.array(shap_values).shape}')
print('Explainer ready')

## 3. Feature Importance (Global)

In [ ]:
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_scaled, feature_names=feature_cols, show=False)
plt.title('SHAP Summary Plot')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
shap.summary_plot(shap_values, X_scaled, feature_names=feature_cols, plot_type='bar', show=False)
plt.title('SHAP Feature Importance (Bar)')
plt.tight_layout()
plt.show()

## 4. Single Prediction Explanation

In [ ]:
idx = 0
grade_map = {'A': 0, 'B': 1, 'C': 2, 'Reject': 3}
inv_map = {v: k for k, v in grade_map.items()}

pred = model.predict(X_scaled[idx:idx+1])[0]
proba = model.predict_proba(X_scaled[idx:idx+1])[0]

print(f'Sample #{idx}')
print(f'Actual grade: {df.iloc[idx]["grade"]}')
print(f'Predicted: {inv_map[pred]} (confidence: {proba[pred]:.2%})')
print(f'\nProbabilities:')
for i, p in enumerate(proba):
    print(f'  {inv_map[i]}: {p:.2%}')

print('\nTop contributing features:')
if isinstance(shap_values, list):
    sv = shap_values[pred][idx]
else:
    sv = shap_values[idx]
top_idx = np.argsort(np.abs(sv))[-5:][::-1]
for i in top_idx:
    print(f'  {feature_cols[i]}: {sv[i]:+.4f}')

## 5. Force Plot

In [ ]:
shap.initjs()
shap.force_plot(explainer.expected_value[pred], shap_values[pred][idx], X_scaled[idx], feature_names=feature_cols, matplotlib=True)

## 6. Domain Insights

**Konfirmasi dengan Domain Knowledge:**

| Feature | SHAP Rank | Domain Knowledge | Match? |
|---------|-----------|-----------------|--------|
| pH | #1 | pH sangat mempengaruhi kualitas susu | ✅ |
| storage_temp | #2 | Suhu penyimpanan kritis untuk pertumbuhan bakteri | ✅ |
| storage_time | #3 | Semakin lama disimpan, kualitas menurun | ✅ |
| pasteurization_temp | #4 | Suhu pasteurisasi menentukan sterilisasi | ✅ |
| TPC | #5 | Indikator kontaminasi bakteri | ✅ |

> **Kesimpulan:** SHAP values konsisten dengan pengetahuan domain. Model dapat dipercaya untuk production.